# **Comprehensive Theory of Multivariate Imputation**

Unlike univariate imputation—which looks blindly at a single column to calculate a replacement value (like the mean)—**Multivariate Imputation** models missing values by analyzing the relationships and statistical patterns **across multiple columns simultaneously**. 

It fundamentally treats missing data handling not as a basic filling routine, but as a **Machine Learning sub-problem**.

---

## 1. Core Mathematical Concept: How it Works

Multivariate imputation operates on the assumption that features in a dataset do not exist in a vacuum; they share mathematical dependencies, correlations, and interactions.

*   **The Predictive Approach:** If a row represents an individual whose `Salary` is missing, a univariate imputer would simply fill it with the average salary of the entire dataset. A multivariate imputer, however, will evaluate that specific individual's `Age`, `Years of Education`, `Job Title`, and `Zip Code`. It uses those known features as predictors to estimate the missing `Salary`.

*   **Missing Data Mechanism:** This approach is uniquely engineered to handle data that is **MAR (Missing At Random)**. This means the missingness is not completely random, but can be fully explained and accounted for by looking at other observed variables in the dataset.

---

## 2. The Core Methodologies Shown in the Mind Map

There are two primary paradigms under multivariate imputation, each utilizing a different algorithmic architecture:

### A. KNN Imputer (Distance-Based Approach)

*   **The Logic:** This approach relies on spatial distance metrics. To fill a missing value in a specific row, the algorithm calculates the geometric distance (typically Euclidean distance) across all other completed features to find the **$K$ most similar rows** (Nearest Neighbors) in the dataset.

*   **The Execution:** Once the $K$ neighbors are identified, it averages their values (for numerical features) or takes the most frequent label (for categorical features) to fill the blank spot.

### B. Iterative Imputer / MICE (Chain Equations Approach)

*   **The Logic:** **MICE** stands for *Multivariate Imputation by Chained Equations*. It turns every single column that contains missing data into a separate machine learning model (e.g., a Linear Regression or a Decision Tree).

*   **The Execution (Round-Robin):** 

    1.  It begins by filling all missing spots with a quick baseline guess (like the median).
    2.  It drops the baseline guesses for Column 1, treats Column 1 as the target variable ($Y$), and trains a model using Columns 2, 3, and 4 as features ($X$) to predict the missing slots.
    3.  It moves to Column 2, treats it as the target variable ($Y$), and trains a model using Columns 1, 3, and 4 as features.
    4.  This cycle repeats iteratively across all columns for multiple rounds (iterations) until the predicted values stabilize.

---

## 3. General Advantages of Multivariate Imputation

### A. Unmatched Predictive Precision
Because it builds context-aware estimates based on surrounding data points, the values injected are highly realistic. It completely avoids the artificial, distribution-destroying "spikes" caused by mean or median imputation.

### B. Preserves Covariance and Complex Interactions
This methodology keeps the true multi-dimensional shape of your data intact. By predicting values that naturally align with other columns, it fully preserves the underlying correlations and joint distributions that linear regressions and tree models rely on.

---

## 4. General Disadvantages and Risks

### A. Extreme Computational and Memory Overhead

Because this strategy requires training machine learning models or calculating vast spatial coordinate matrices just to clean the data, it is exceptionally resource-heavy. 
*   **Impact:** It can cause pipeline speeds to crawl to a halt on large datasets, significantly increasing training costs and infrastructure requirements.

### B. Severe Risk of Data Leakage (If Misconfigured)
If a multivariate imputer is mistakenly fit on the entire dataset *before* performing a Train-Test split, your training features will absorb precise structural patterns from the validation/test rows. This leaks test data into training, leading to highly inflated, unrealistic cross-validation scores.

### C. Live Production Scoring Latency
During a real-time production deployment (e.g., scoring a single user profile streaming into a credit-check app), running a complex sub-model like KNN or a chained equation just to fill a blank field introduces significant scoring latency. This makes it challenging to use in strict real-time, low-latency environments.


In [1]:
import numpy as np
import pandas as pd

# **Deep Dive: KNN Imputer (K-Nearest Neighbors)**

The **KNN Imputer** is a multivariate imputation technique that uses spatial distance metrics to fill missing values. Instead of calculating a single global statistic (like the mean) for a column, it looks at the surrounding feature space to find the **K most similar rows** (neighbors) and borrows their values to fill the missing gaps.

---

## 1. Core Mechanics: How It Works

When a row contains a missing value, the algorithm calculates its geometric proximity to all other rows in the dataset using a modified version of **Euclidean Distance**.

### A. The Distance Formula with Missing Values
Standard Euclidean distance breaks down if features are missing. To bypass this, the KNN Imputer utilizes an adjusted formula that ignores missing attributes and scales up the remaining weights(nan Euclidean distance):

$$d(x, y) = \sqrt{\frac{\text{Total Features}}{\text{Observed Features}} \sum_{i \in \text{Observed}} (x_i - y_i)^2}$$

*   **Adjustment:** If two rows are being compared across 4 columns, but one column is missing in row x, the algorithm calculates the distance using the 3 known columns and multiplies the result by $\frac{4}{3}$ to normalize the scale.

### B. The Imputation Step

1.  **Find Neighbors:** The algorithm identifies the K rows (default is usually `n_neighbors=5`) that have the smallest calculated distance to the target row.

2.  **Aggregate Values:** 
    *   **Numerical Features:** It calculates the mean or a weighted average of the missing feature across those K neighbors.
    *   **Categorical Features:** It takes the mode (most frequent label) among the neighbors.

### Scikit-Learn Implementation
```python
from sklearn.impute import KNNImputer

# Initialize the imputer
# weights='uniform' treats all neighbors equally. 
# weights='distance' gives closer neighbors a stronger vote.
knn_imputer = KNNImputer(n_neighbors=5, weights='uniform')

# Example Execution:
# X_train_imputed = knn_imputer.fit_transform(X_train)
```

---

## 2. Prerequisites for Use (When to Apply)

1.  **Strong Feature Correlations:** The dataset must contain columns that naturally share physical or logical relationships (e.g., if `Weight` is missing, it correlates heavily with `Height`, `Age`, and `Gender`). If your features are completely independent, neighbor calculations will yield meaningless values.

2.  **Missing At Random (MAR):** The missing data should be explainable by the other observed features in the row.

3.  **Small to Medium-Sized Datasets:** The data volume must be compact enough to fit comfortably within system memory allocations.

---

## 3. Advantages (Benefits)

### A. Context-Aware Local Accuracy
Unlike a global mean that assigns everyone the same average, the KNN Imputer generates highly personalized estimations. For example, if a 6-foot-4 athlete's weight is missing, it will average the weights of other tall, active individuals rather than pulling down the average using data from shorter, sedentary individuals.

### B. Preserves Complex Data Boundaries
Because it relies on localized clusters, it naturally respects multi-dimensional shapes, curves, and non-linear boundaries within your dataset. This makes it an exceptional preprocessor for non-linear models.

---

## 4. Disadvantages and Risks (Why to Avoid)

### A. Massive Memory and Computational Footprint
The KNN algorithm is an "instance-based" or lazy learner. It doesn't actually learn a compact mathematical equation during training; instead, it stores the entire dataset.

*   **The Computation Bottleneck:** For every single missing value, it must compute spatial distances against **every other row** in your dataset. If you have $N$ rows and $M$ columns, the time complexity scales at $\mathcal{O}(N^2 \cdot M)$. On massive datasets, your pipeline will easily crash or freeze due to memory limits.

### B. Mandatory Requirement for Feature Scaling
Because Euclidean distance relies entirely on absolute numerical differences, features with larger scales will completely dominate the neighbor calculation.

*   **The Scale Problem:** If `Salary` ranges from \$30,000 to \$200,000 and `Age` ranges from 18 to 80, the massive numerical spreads in `Salary` will completely drown out `Age` variations during distance calculations. You **must** scale your features (e.g., using `MinMaxScaler` or `StandardScaler`) *before* running a KNN Imputer.

---

## 5. Model Compatibility Matrix

| Algorithm Family | Compatibility Status | Core Reason |
| :--- | :--- | :--- |
| **Distance-Based (KNN, K-Means, SVM)** | ✅ Highly Recommended | Perfectly aligns with the structural geometric assumptions of the final estimator models. |
| **Linear Models (Linear/Logistic Regression)** | ✅ Compatible | Safely preserves the natural slope dynamics and underlying correlations across columns. |
| **Tree-Based Models (Random Forests, XGBoost)** | ⚠️ Neutral | While accurate, tree models can natively handle missing values or utilize faster methods. The extreme compute cost of KNN is often a waste of time for tree pipelines. |
| **Big Data / Real-Time Live Deployments** | ❌ Avoid | Calculating matrix distances on streaming production records introduces severe latency bottlenecks, breaking real-time response constraints. |


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer, KNNImputer

In [3]:
# Import the dataset
df = pd.read_csv("../../data/titanic.csv", usecols=['Age', 'Pclass', 'Fare', 'Survived'])
df.sample(5)

,Survived,Pclass,Age,Fare
620,0,3,27.0,14.4542
716,1,1,38.0,227.5250
518,1,2,36.0,26.0000
664,1,3,20.0,7.9250
890,0,3,32.0,7.7500


In [4]:
# check for Missing values percentage
df.isnull().mean()*100

Survived     0.00000
Pclass       0.00000
Age         19.86532
Fare         0.00000
dtype: float64

In [5]:
# Split the data into features and target
X = df.drop('Survived', axis=1)
y = df['Survived']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=4)
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")

Shape of X_train: (712, 3)
Shape of X_test: (179, 3)


### 1. KNN Impumputer 

In [6]:
# Fill the missing values using KNNImputer
knn = KNNImputer(n_neighbors=5, weights='distance')
X_train_imputed = knn.fit_transform(X_train)
X_test_imputed = knn.fit_transform(X_test)

# Convert back to pandas dataframe
X_train_imputed = pd.DataFrame(X_train_imputed, columns=X_train.columns)
X_test_imputed = pd.DataFrame(X_test_imputed, columns=X_test.columns)

# check the first 5 rows of the imputed data
X_train_imputed.sample(5)

,Pclass,Age,Fare
1,2.0,60.000000,39.0000
629,3.0,49.000000,0.0000
198,3.0,31.000000,8.6833
401,3.0,20.000000,8.0500
358,3.0,24.209262,7.7875


### 2. Simple Imputer

In [7]:
# Fill the missing values using SimpleImputer
si = SimpleImputer(strategy='median')
X_train_SiImputed = si.fit_transform(X_train)
X_test_SiImputed = si.fit_transform(X_test)

# Convert back to pandas dataframe
X_train_SiImputed = pd.DataFrame(X_train_SiImputed, columns=X_train.columns)
X_test_SiImputed = pd.DataFrame(X_test_SiImputed, columns=X_test.columns)

# check the first 5 rows of the imputed data
X_train_SiImputed.sample(5)

,Pclass,Age,Fare
204,2.0,34.0,21.0000
291,3.0,19.0,7.8542
88,2.0,18.0,11.5000
308,1.0,28.5,25.9250
440,2.0,48.0,65.0000


In [8]:
# Training Model
def train_model(X_train, X_test, y_train, y_test):
    
    # Train a logistic regression model on the imputed data
    model = LogisticRegression()

    # Evaluate the model
    model.fit(X_train, y_train)

    # Make predictions on the test set
    y_pred = model.predict(X_test)

    # Accuracy of the model
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy of the logistic regression model: {accuracy:.4f}")



In [9]:
# Model Accuracy on KNN Imputed data
train_model(X_train_imputed, X_test_imputed, y_train, y_test)

# When weight = 'uniform' and n_neighbors = 5  : accuracy is : 0.7374 (Gives Equal weight to each neighbor)
# When weight = 'distance' and n_neighbors = 5  : accuracy is : 0.7486 (Gives more weight to neighbors that are closer)

Accuracy of the logistic regression model: 0.7486


> Observations: 
- In General, the KNN model with 'distance' weighting and 5 neighbors tends to perform better than the Logistic Regression model. 
- This is because the K NN model can handle non-linear relationships between features more effectively than logistic regression. 
- The distance-based weighting allows the model to consider the proximity of neighboring data points when making predictions, which can help it capture complex patterns in the data. 

In [10]:
# Model Accuracy on SimpleImputer Imputed data
train_model(X_train_SiImputed, X_test_SiImputed, y_train, y_test)


Accuracy of the logistic regression model: 0.7374


> Conclusion: 
- Most of the time KNN imputer outperforms SimpleImputer for predicting the target variable in this dataset. As the KNN imputer calculates the relationship between features and target variable more accurately than SimpleImputer so, it can provide better predictions. 

# ============================================================

# **Deep Dive: Iterative Imputer / MICE (Multivariate Imputation by Chained Equations)**

**Iterative Imputer** (popularly known as **MICE** in the broader data science community) represents the gold standard for multivariate imputation. Instead of treating missing data handling as a simple statistical patch, it converts the process into a series of **Machine Learning prediction sub-problems**, modeling every single column with missing values as a function of all other columns in a round-robin loop.

---

## 1. Core Mechanics: How It Works

Imagine you have a dataset with three columns that contain missing values: `Age`, `Income`, and `Credit_Score`. The algorithm resolves these gaps through an iterative process:

### Step 1: Initial Baseline Fill
The imputer starts by filling every `NaN` value across all columns with a quick placeholder guess, such as the column's **median** or **mean**. This creates a fully filled temporary dataset.

### Step 2: The Round-Robin Modeling Cycle (Chained Equations)
The algorithm strips away the placeholder guesses for **one specific column** at a time and treats it as a machine learning target:

1.  **Imputing `Age`:** It removes the placeholders from the `Age` column. It sets `Age` as the target variable (Y) and trains a machine learning model (e.g., a Bayesian Ridge Regression or Decision Tree) using `Income` and `Credit_Score` as the feature matrix (X). It then predicts and fills the missing `Age` values.

2.  **Imputing `Income`:** It removes placeholders from `Income`. It sets `Income` as the target (Y) and trains a new model using the newly updated `Age` values and `Credit_Score` as features (X). It predicts and fills the missing `Income` values.

3.  **Imputing `Credit_Score`:** It repeats the process, setting `Credit_Score` as the target (Y) and using `Age` and `Income` as features (X).

### Step 3: Iteration and Convergence

This complete sequence represents **one iteration**. The algorithm repeats this entire cycle for multiple rounds (configured via `max_iter`, typically set to 10). With each round, the prediction models become sharper, and the imputed values adjust until they stabilize (**converge**), matching the true multi-dimensional distribution of the data.

### Scikit-Learn Implementation
In scikit-learn, this class is still considered experimental, meaning you must explicitly enable it before importing.

```python
# 1. You must explicitly enable the experimental imputer first
from sklearn.experimental import enable_iterative_imputer  
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge

# 2. Initialize the imputer
# estimator specifies which ML model is built under the hood for each column loop
iterative_imputer = IterativeImputer(
    estimator=BayesianRidge(), 
    max_iter=10, 
    random_state=42
)

# Example Execution:
# X_train_imputed = iterative_imputer.fit_transform(X_train)
```

---

## 2. Prerequisites for Use (When to Apply)

1.  **MAR (Missing At Random):** The missing data should share a logical relationship with other columns. For instance, if an individual's `Income` is missing, it can be accurately inferred by examining their `Years of Experience`, `Education Level`, and `Zip Code`.

2.  **Complex, Multi-Feature Data Patterns:** Best applied when downstream models rely heavily on intricate correlations, interactions, and strict linear/non-linear dependencies across features.

---

## 3. Advantages (Benefits)

### A. Unmatched Imputation Precision
Because it builds a custom, dedicated machine learning model for every single missing column, the generated values are highly sophisticated, realistic, and tailored to the exact context of that specific row.

### B. Full Preservation of Covariance & Correlations
Unlike univariate techniques that dilute correlations, the Iterative Imputer naturally respects and preserves the underlying linear relationships and joint distributions across variables. This prevents distribution shapes from breaking down.

### C. Algorithmic Flexibility
Scikit-learn allows you to swap out the underlying `estimator`. If your data contains non-linear relationships, you can pass a `RandomForestRegressor` or an `ExtraTreesRegressor` as the estimator, enabling the imputer to capture complex, non-linear patterns.

---

## 4. Disadvantages and Risks (Why to Avoid)

### A. High Computational Cost
Training multiple machine learning sub-models across dozens of features for 10 iterations requires a massive amount of CPU cycles and memory. On exceptionally large datasets, training an Iterative Imputer can take longer than training the final production model itself.

### B. High Risk of Data Leakage (If Misconfigured)
If `fit_transform` is accidentally run on your entire dataset before splitting it into Train and Test sets, the sub-models will learn the structural patterns of your test set. This completely contaminates your validation process, leading to heavily inflated and unrealistic accuracy scores.

---

## 5. Model Compatibility Matrix

| Algorithm Family | Compatibility Status | Core Reason |
| :--- | :--- | :--- |
| **Linear & Logistic Regression** | ✅ Highly Recommended | Preserves column correlations and data variance, keeping coefficient weights completely stable. |
| **Neural Networks (Deep Learning)** | ✅ Highly Recommended | Generates smooth, realistic feature values that prevent gradients from exploding or vanishing due to artificial data spikes. |
| **Tree-Based Models (Random Forests, XGBoost)** | ⚠️ Neutral | While it provides highly accurate data, tree-based models are robust enough to handle simpler techniques (like arbitrary value imputation) just as effectively without the severe computational slowdown of MICE. |
| **Low-Latency Production API Pipelines** | ❌ Avoid | Running a chained sequence of machine learning sub-models to impute missing fields on a real-time incoming user request introduces massive scoring latency. |


In [11]:
# Enabling experimental imputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge


In [12]:
# Initializing the imputer
iterative_imputer = IterativeImputer(
    estimator=BayesianRidge(),
    max_iter=100,
    tol=1e-1,
    initial_strategy ='median',
    random_state=45,
)

# Fitting the imputer to the training data
X_train_ItImputed = iterative_imputer.fit_transform(X_train)
X_test_ItImputed = iterative_imputer.fit_transform(X_test)

# Converting back to DataFrame 
X_train_ItImputed = pd.DataFrame(X_train_ItImputed, columns=X_train.columns)
X_test_ItImputed = pd.DataFrame(X_test_ItImputed, columns=X_test.columns)


In [13]:
# Checking the first few rows of the imputed data
X_train_ItImputed.sample(5)

,Pclass,Age,Fare
229,2.0,29.0,21.0000
1,2.0,60.0,39.0000
589,3.0,16.0,20.2500
112,2.0,28.0,12.6500
343,1.0,58.0,146.5208


In [14]:
# Training and Testing the Model on Imputed Data
train_model(X_train_ItImputed, X_test_ItImputed, y_train, y_test)

Accuracy of the logistic regression model: 0.7374


# The Core Foundation: Missing Data Mechanisms (MCAR, MAR, MNAR)

To choose the perfect missing data handling strategy, you must understand **why** the data is missing in the first place. Statistically, all missing data falls into one of three distinct mathematical frameworks: **MCAR**, **MAR**, or **MNAR**. 

---

## 1. MCAR: Missing Completely At Random

### What It Is
The missingness has absolutely nothing to do with any value in the dataset—either observed or unobserved. The missing data points are a purely random, accidental subset of the entire data matrix.

*   **Real-World Example:** A test tube drops and shatters in a lab, destroying a blood sample. Or, a bad internet connection causes a random packet drop during a survey submission.

*   **The Statistical Reality:** The observed data remains an unbiased, perfect miniature representation of the complete population distribution.

### Best Imputation Choices for MCAR

Because the gaps are completely random, simple statistical metrics calculated from the rest of the column are completely unbiased.

*   **Complete Case Analysis (Deletion):** Highly effective if missing data is under 5%. Dropping rows will not introduce statistical bias.
*   **Mean / Median Imputation:** Safe to use. It gives a fast baseline without shifting the true data center.
*   **Random Sample Imputation:** Outstanding choice for linear/distance models because it fills gaps while perfectly keeping the original column variance and distribution shape intact.

---

## 2. MAR: Missing At Random

### What It Is
The missingness is **not** random across the whole dataset, but it can be **completely explained and accounted for** by looking at *other* available columns (observed features) in the dataset.

*   **Real-World Example:** In a public opinion survey, men are statistically less likely to answer questions about their emotional well-being than women. The missingness in the `Emotional_Score` column is directly linked to the `Gender` column.

*   **The Statistical Reality:** If you isolate just the male rows, the missingness within that sub-group *becomes* completely random.

### Best Imputation Choices for MAR
Univariate methods (like Mean/Median) fail spectacularly here because they ignore the cross-column relationships causing the missingness. You **must** use techniques that look across multiple features.

*   **Iterative Imputer (MICE):** The **absolute gold standard** for MAR. It builds sub-models to estimate the missing value based precisely on those correlating columns (e.g., using `Gender` and `Age` to predict `Emotional_Score`).

*   **KNN Imputer:** Highly effective. It locates the closest geographic neighbors who share identical observed features and steals their values to fill the gap.

*   **Mode Imputation (Categorical only):** Can be used within grouped subsets (e.g., imputing the missing mode value of a column *after* grouping by a known category).

---

## 3. MNAR: Missing Not At Random

### What It Is
The missingness depends entirely on the **unobserved value itself**. The reason the data is missing is directly tied to what the hidden value would have been.

*   **Real-World Example:** In a financial survey, individuals with exceptionally high salaries intentionally leave the `Salary` field blank due to privacy concerns. Or, a failing student skips the final exam entirely.
*   **The Statistical Reality:** The data is systematically biased. The observed records are *not* representative of the missing records, making it impossible for standard math to guess the hidden numbers accurately.

### Best Imputation Choices for MNAR
Because the missingness holds deep structural meaning, trying to guess the "true" value using standard statistics will corrupt your model. Instead, your goal must be to **flag the missingness** so the machine learning model can learn from the pattern of omission.

*   **Missing Indicator Flag:** The absolute best strategy. You create a binary `0/1` column tracking exactly what was missing, allowing algorithms to see that an omission has occurred.

*   **Constant / "Missing" Label Imputation:** Excellent for text data (e.g., filling empty cells with `"Not Disclosed"`). This treats the omission as its own independent category.

*   **End of Distribution Imputation:** Best for tree-based models. Forcing missing cells into a massive outlier tail value (like `Mean + 3*Std`) allows tree splits to quickly segregate and isolate the missing behavior.

---

## 4. Summary Selection Matrix

| Missing Mechanism | Underlying Cause | Data Bias Level | Recommended Techniques | Techniques to Avoid |
| :--- | :--- | :--- | :--- | :--- |
| **MCAR** *(Completely At Random)* | Pure accident / equipment glitch | 🟢 None | Deletion ($<5\%$), Mean, Median, Random Sampling | Arbitrary Value Imputation (`-999`) |
| **MAR** *(At Random)* | Explained by other columns | 🟡 Conditional | **KNN Imputer**, **Iterative Imputer (MICE)** | Mean, Median, Row Deletion (introduces severe bias) |
| **MNAR** *(Not At Random)* | Hidden value itself dictates omission | 🔴 Severe | **Missing Indicator**, Constant Label, End of Distribution | Mean, Median, KNN, MICE (all yield warped predictions) |
